# BusLight — Pipeline RAG e avaliação

Este notebook apresenta a base documental, as métricas de recuperação e o placar da execução do LLM salvo em `avaliacao/`. A inferência foi executada localmente com Ollama; este notebook **lê os resultados já salvos**, não simula respostas.

In [1]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd()))
from src.knowledge import load_jsonl
from src.retrieval import Retriever
chunks = load_jsonl()
print('Trechos na base:', len(chunks))
print('Exemplo:', chunks[0]['chunk_id'], chunks[0]['source_id'])

Trechos na base: 16
Exemplo: BL-001 BUSLIGHT_INTERNO


In [2]:
from src.evaluate_retrieval import read_cases
cases = read_cases()
print('Casos totais:', len(cases))
print('Perguntas respondíveis:', sum(bool(c['relevant_chunk_ids']) for c in cases))
print('Perguntas para abstenção:', sum(c['should_abstain'] for c in cases))

Casos totais: 21
Perguntas respondíveis: 16
Perguntas para abstenção: 5


In [3]:
retriever = Retriever(chunks)
metrics = retriever.metrics(cases)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "recall@1": 0.8333333333333334,
  "hit_rate@1": 0.9375,
  "recall@3": 1.0,
  "hit_rate@3": 1.0,
  "answerable_queries": 16,
  "total_relevant_labels": 18
}


In [4]:
for cid in ['Q02','Q06','Q17','Q21']:
    case=next(c for c in cases if c['id']==cid)
    hits=retriever.retrieve(case['question'], top_k=3)
    print(cid, case['question'])
    print([(x['chunk_id'], round(x['score'],4)) for x in hits])

Q02 De onde vêm as posições dos ônibus em tempo real?
[('SPT-002', 0.2997), ('GTFS-001', 0.1078), ('BL-002', 0.104)]
Q06 No GTFS, o que significa um horário 25:35:00?
[('GTFS-002', 0.3286), ('GTFS-001', 0.0864), ('SPT-002', 0.0809)]
Q17 Qual é a lotação exata do ponto da Rua das Flores agora?
[('BL-003', 0.1526), ('BL-002', 0.0983), ('SPT-001', 0.086)]
Q21 Qual é o mascote oficial do BusLight e qual é a cor dele?
[('BL-003', 0.1127), ('BL-002', 0.1012), ('SPT-005', 0.084)]


## Comparação das três configurações

A execução salva no repositório contém 21 perguntas em cada modo (63 respostas): `base`, `rag` e `agent`. O modelo usado foi `qwen2.5:3b`.

A célula seguinte lê o CSV efetivamente gerado pelo benchmark. Se o código, o prompt ou o conjunto de testes forem alterados, execute novamente `python -m src.evaluate_llm` e reexecute este notebook para atualizar as evidências.

In [6]:
from pathlib import Path
import csv

placar_path = Path("avaliacao/placar_tres_configuracoes.csv")
with placar_path.open(encoding="utf-8-sig", newline="") as handle:
    placar = list(csv.DictReader(handle, delimiter=";"))

for row in placar:
    print(
        f"{row['mode']}: aprovação={float(row['accuracy_pass_rate']):.1%}; "
        f"JSON={float(row['json_valid_rate']):.1%}; "
        f"citações={row['citation_rate'] or 'N/A'}; "
        f"abstenção={float(row['abstention_accuracy']):.1%}; "
        f"latência={float(row['mean_latency_seconds']):.2f}s"
    )

# Confirma que o arquivo bruto tem 63 registros (21 perguntas x 3 modos).
raw_path = Path("avaliacao/respostas_benchmark.jsonl")
raw_count = sum(1 for line in raw_path.read_text(encoding="utf-8").splitlines() if line.strip())
print("Respostas brutas registradas:", raw_count)


base: aprovação=0.0%; JSON=100.0%; citações=N/A; abstenção=0.0%; latência=11.16s
rag: aprovação=19.0%; JSON=100.0%; citações=62.5%; abstenção=80.0%; latência=31.21s
agent: aprovação=61.9%; JSON=100.0%; citações=93.8%; abstenção=100.0%; latência=32.74s
Respostas brutas registradas: 63


## Auditoria de falhas observadas

Os casos abaixo foram extraídos das respostas salvas. São erros observados no benchmark; as correções propostas ainda precisam ser testadas com nova execução.

In [7]:
import json
from pathlib import Path

records = [json.loads(line) for line in Path("avaliacao/respostas_benchmark.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
selected = [("base", "Q10"), ("rag", "Q01"), ("rag", "Q07"), ("agent", "Q15"), ("rag", "Q19")]
for mode, case_id in selected:
    record = next(row for row in records if row["mode"] == mode and row["case_id"] == case_id)
    print(f"\n{mode.upper()} {case_id} | pass={record['pass']} | status={record['status']}")
    print("Pergunta:", record["question"])
    print("Resposta literal:", record["answer"] or "[campo answer vazio]")
    print("Citações:", record["citations"])


BASE Q10 | status=answered | pass=False
Pergunta: Quais princípios da LGPD o BusLight deve seguir?
Resposta: O BusLight deve seguir os princípios da LGPD (Lei Geral sobre a Privacidade e Proteção de Dados), que incluem: Objetividade, Limite de Coleta, Transparência, Limitação de Dados, Integridade e Qualidade, Proporção, Não Dano, Autonomia do Titular dos Dados, Respeito ao Direito de Oposição e Direito de Apagamento.
Citações: []

RAG Q01 | status=answered | pass=False
Pergunta: Como o BusLight deve informar o tempo estimado de chegada?
Resposta: CHUNK_ID: SPT-003 | CHUNK_ID: BL-003 | CHUNK_ID: SPT-002
Citações: [{"chunk_id": "SPT-003", "source_id": "SPTRANS_OLHOVIVO", "title": "Previsão de chegada", "url": ""}, {"chunk_id": "BL-003", "source_id": "BUSLIGHT_INTERNO", "title": "Regra de fundamentação e previsão", "url": ""}, {"chunk_id": "SPT-002", "source_id": "SPTRANS_OLHOVIVO", "title": "Posição dos veículos", "url": ""}]

RAG Q07 | status=abstained | pass=False
Pergunta: Como a API